# L1_T5 — Fine-Tuning on Public, Dermatologist-Labeled Data (SCIN)
### Layer 1 of the Aura pipeline · Task 5 of N

**What changed from the original plan:** the original T5 assumed 200+ consented Filipino volunteers labeled by a
dermatologist partner. That data doesn't exist yet, so this version fine-tunes the Fitzpatrick head on a
**public dataset that allows this use**: Google's **SCIN** (Skin Condition Image Network, `google/scin` on
Hugging Face), which has dermatologist-estimated Fitzpatrick labels and self-reported race/ethnicity.

**What this does and doesn't give you (please read):**
- ✅ More real, dermatologist-labeled examples of Fitzpatrick III–VI, plus an **Asian-flagged subset** you can
  evaluate separately.
- ❌ It is **not** Filipino-specific. SCIN is US-based and "Asian" is not broken down further. Results for the
  Asian subset must be reported as "Asian subset of SCIN", never as "Filipino accuracy".
- ❌ SCIN has **no undertone labels**, so this notebook trains the Fitzpatrick head only.
- ⚠️ Most SCIN photos show skin conditions on body parts, not faces. The preprocessing step already falls back
  to the whole image when no face is found.

**Data rules baked into this notebook:**
- **MST-E is never used for training.** Its terms allow research, evaluation and annotator training only, not
  training ML models. It stays as the held-out bias check in `L1_T4`.
- SCIN's license requires attribution, a note of modifications, and **no re-identification** of contributors.
  Step 2 writes a notice file recording this.
- This is not legal advice. Have someone qualified confirm SCIN's terms fit Aura's commercial use.
- Open item outside this notebook: confirm Fitzpatrick17k's license (used for Stage C in `L1_T3`) permits
  commercial use.

## Step 0 — Mount Drive and confirm prior stages exist

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_ROOT = "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone"
MANIFEST_PATH = f"{DRIVE_ROOT}/manifest.json"
DRIVE_CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

STAGE_C_CKPT = os.path.join(DRIVE_CKPT_DIR, "stage_c_best.weights.h5")
STAGE_D_CKPT = os.path.join(DRIVE_CKPT_DIR, "stage_d_best.weights.h5")
assert os.path.exists(STAGE_C_CKPT), f"Missing {STAGE_C_CKPT} -- did L1_T3 finish?"
assert os.path.exists(STAGE_D_CKPT), f"Missing {STAGE_D_CKPT} -- did L1_T3's Stage D addition finish?"
print("✅ Stage C (Fitzpatrick) and Stage D (undertone) checkpoints both found.")

if "mste_bias_check" in manifest:
    print("\nL1_T4 bias check summary (context for this notebook):")
    print(f"  Overall mean error: {manifest['mste_bias_check']['overall_mean_error']}")
    print(f"  Images evaluated: {manifest['mste_bias_check']['images_evaluated']}")
else:
    print("\n(L1_T4 hasn't been run yet -- not required to proceed here, just noting it's missing.)")


Mounted at /content/drive
✅ Stage C (Fitzpatrick) and Stage D (undertone) checkpoints both found.

L1_T4 bias check summary (context for this notebook):
  Overall mean error: 0.429
  Images evaluated: 1488


## Step 1 — Record the data source and license obligations
Writing this down up front means every later step (and any future audit) can trace where the training data
came from and under what terms.

In [ ]:
DRIVE_DATA_DIR = f"{DRIVE_ROOT}/datasets/scin"
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

DATA_SOURCE = {
    "name": "SCIN (Skin Condition Image Network)",
    "hf_id": "google/scin",
    "page": "https://huggingface.co/datasets/google/scin",
    "license": "SCIN Data Use License",
    "citation": "Ward et al., JAMA Netw Open 2024, doi:10.1001/jamanetworkopen.2024.46615",
    "labels_used": "dermatologist_fitzpatrick_skin_type_label_1..3 (estimated Fitzpatrick)",
    "obligations": [
        "Keep attribution: creator, copyright notice, link to the SCIN license and dataset.",
        "State that images were modified (face-crop / resize / CLAHE) if sharing derived data.",
        "Never attempt to re-identify or re-link contributors.",
    ],
    "excluded": "MST-E (terms prohibit training ML models; evaluation only, see L1_T4)",
}
for k, v in DATA_SOURCE.items():
    print(f"{k}: {v}")

name: SCIN (Skin Condition Image Network)
hf_id: google/scin
page: https://huggingface.co/datasets/google/scin
license: SCIN Data Use License
citation: Ward et al., JAMA Netw Open 2024, doi:10.1001/jamanetworkopen.2024.46615
labels_used: dermatologist_fitzpatrick_skin_type_label_1..3 (estimated Fitzpatrick)
obligations: ['Keep attribution: creator, copyright notice, link to the SCIN license and dataset.', 'State that images were modified (face-crop / resize / CLAHE) if sharing derived data.', 'Never attempt to re-identify or re-link contributors.']
excluded: MST-E (terms prohibit training ML models; evaluation only, see L1_T4)


## Step 2 — Write the license/attribution notice next to the data

In [ ]:
notice = f"""# Data provenance notice (L1_T5)

Source: {DATA_SOURCE['name']} -- {DATA_SOURCE['page']}
License: {DATA_SOURCE['license']} (full text on the dataset page)
Citation: {DATA_SOURCE['citation']}

Obligations carried by this project:
""" + "\n".join(f"- {o}" for o in DATA_SOURCE["obligations"]) + f"""

Modifications made: face-detect crop (or whole image if no face), resize to 224x224, CLAHE on L channel.

Explicitly NOT used for training: {DATA_SOURCE['excluded']}.
"""
notice_path = f"{DRIVE_DATA_DIR}/DATA_PROVENANCE.md"
with open(notice_path, "w") as f:
    f.write(notice)
print(f"✅ Provenance notice written: {notice_path}")

✅ Provenance notice written: /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/datasets/scin/DATA_PROVENANCE.md


## Step 3 — Load SCIN, build usable labels, and gate on how much there is
Keeps only cases that have at least one dermatologist Fitzpatrick label. If a case has several, they must agree
within 1 point (otherwise the case is dropped as ambiguous), and the label used is the median, rounding .5 up.
One image per case is used, so the same person never lands in both train and validation.

The gate still exists, but it now checks real label counts. It stops with a clear error if SCIN filtering
leaves too little to fine-tune on, and warns if the Asian subset is too small to evaluate on.

In [ ]:
!pip install -q datasets

import numpy as np
import pandas as pd
from datasets import load_dataset

scin = load_dataset("google/scin", split="train")
print(f"SCIN cases loaded: {len(scin)}")

IMAGE_COLS = ["image_1_path", "image_2_path", "image_3_path"]
meta = scin.select_columns([c for c in scin.column_names if c not in IMAGE_COLS]).to_pandas()
meta["dataset_index"] = np.arange(len(meta))

MIN_DERM_LABELS = 1      # raise to 2 for cleaner (but fewer) labels
MAX_LABEL_SPREAD = 1     # multiple dermatologists must agree within this many points
MINIMUM_TO_PROCEED = 30
MINIMUM_ASIAN_FOR_EVAL = 30

def parse_fst(v):
    if isinstance(v, str) and v.startswith("FST") and v[3:].isdigit():
        n = int(v[3:])
        return n if 1 <= n <= 6 else None
    return None

def reconcile(row):
    labels = [parse_fst(row.get(f"dermatologist_fitzpatrick_skin_type_label_{i}")) for i in (1, 2, 3)]
    labels = [l for l in labels if l is not None]
    if len(labels) < MIN_DERM_LABELS or not labels:
        return None
    if max(labels) - min(labels) > MAX_LABEL_SPREAD:
        return None
    return int(np.floor(np.median(labels) + 0.5))

def is_yes(v):
    return v is True or (isinstance(v, str) and v.strip().upper() in ("YES", "TRUE"))

meta["fitzpatrick_scale"] = meta.apply(reconcile, axis=1)
meta["is_asian"] = meta["race_ethnicity_asian"].map(is_yes)

has_any_label = meta[[f"dermatologist_fitzpatrick_skin_type_label_{i}" for i in (1, 2, 3)]].notna().any(axis=1)
usable_df = meta[meta["fitzpatrick_scale"].notna()].copy()
usable_df["fitzpatrick_scale"] = usable_df["fitzpatrick_scale"].astype(int)
usable_df["case_id"] = usable_df["case_id"].astype(str)
usable_df = usable_df[["case_id", "dataset_index", "fitzpatrick_scale", "is_asian"]].reset_index(drop=True)
usable_ids = set(usable_df["case_id"])

print(f"Cases with any dermatologist Fitzpatrick label: {int(has_any_label.sum())}")
print(f"Dropped as ambiguous (dermatologists disagreed): {int(has_any_label.sum()) - len(usable_df)}")
print(f"\n✅ USABLE labeled cases: {len(usable_df)}")
print(f"   ...of which Asian-flagged: {int(usable_df['is_asian'].sum())}")
print("\nFitzpatrick distribution (all usable):")
print(usable_df["fitzpatrick_scale"].value_counts().sort_index().to_string())
print("\nFitzpatrick distribution (Asian-flagged):")
print(usable_df[usable_df["is_asian"]]["fitzpatrick_scale"].value_counts().sort_index().to_string())

TARGET = MINIMUM_ASIAN_FOR_EVAL
if len(usable_df) < MINIMUM_TO_PROCEED:
    raise RuntimeError(
        f"Only {len(usable_df)} usable labeled cases after filtering (floor is {MINIMUM_TO_PROCEED}). "
        "Check that the dataset loaded fully and that the dermatologist label column names match the schema."
    )
if int(usable_df["is_asian"].sum()) < MINIMUM_ASIAN_FOR_EVAL:
    print(f"\n⚠️  Only {int(usable_df['is_asian'].sum())} Asian-flagged cases (< {MINIMUM_ASIAN_FOR_EVAL}). "
          "Training can proceed, but Step 7's Asian-subset numbers will be too noisy to trust.")
else:
    print("\n✅ Enough data to proceed.")

README.md:   0%|          | 0.00/24.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Some datasets params were ignored: ['splits', 'download_size', 'dataset_size']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

data/train-00000-of-00026.parquet: reconstructing file:   0%|          |  0.00B /  493MB            

...

data/train-00025-of-00026.parquet: reconstructing file:   0%|          |  0.00B /  487MB            

data/train-00025-of-00026.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5033 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]

SCIN cases loaded: 5033
Cases with any dermatologist Fitzpatrick label: 4370
Dropped as ambiguous (dermatologists disagreed): 173

✅ USABLE labeled cases: 4197
   ...of which Asian-flagged: 80

Fitzpatrick distribution (all usable):
fitzpatrick_scale
1     329
2    1373
3    1352
4     713
5     362
6      68

Fitzpatrick distribution (Asian-flagged):
fitzpatrick_scale
1     2
2    18
3    26
4    27
5     7

✅ Enough data to proceed.


## Step 4 — Export the usable images to local disk
Writes one JPEG per usable case (first available of the case's up to 3 photos). Cases whose images are missing
are dropped, and the usable count is re-checked afterwards.

In [ ]:
import shutil
from tqdm import tqdm

LOCAL_ROOT = "/content/aura_local/Layer1_SkinTone"
LOCAL_BETA_IMAGES = f"{LOCAL_ROOT}/scin_images"
os.makedirs(LOCAL_BETA_IMAGES, exist_ok=True)

exported = []
for row in tqdm(usable_df.itertuples(index=False), total=len(usable_df)):
    rec = scin[int(row.dataset_index)]
    img = next((rec[k] for k in IMAGE_COLS if rec.get(k) is not None), None)
    if img is None:
        continue
    img.convert("RGB").save(os.path.join(LOCAL_BETA_IMAGES, f"{row.case_id}.jpg"), "JPEG", quality=95)
    exported.append(row.case_id)

usable_df = usable_df[usable_df["case_id"].isin(exported)].reset_index(drop=True)
usable_ids = set(usable_df["case_id"])
print(f"Exported {len(usable_df)} images "
      f"({int(usable_df['is_asian'].sum())} Asian-flagged).")
assert len(usable_df) >= MINIMUM_TO_PROCEED, "Too few images survived export -- check image columns."

100%|██████████| 4197/4197 [04:03<00:00, 17.25it/s]

Exported 4197 images (80 Asian-flagged).


## Step 5 — Preprocess (same pipeline as L1_T2, applied to the new images)

Identical face-detect-with-fallback + CLAHE normalization as the rest of this series, so the fine-tuning data
matches the distribution the model already expects.

In [ ]:
!pip install -q mediapipe opencv-python-headless pillow pandas scikit-learn tqdm tensorflow

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from PIL import Image

IMG_SIZE = 224
model_path = "/content/aura_local/blaze_face_short_range.tflite"
if not os.path.exists(model_path):
    import urllib.request
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
        model_path,
    )
detector = mp_vision.FaceDetector.create_from_options(
    mp_vision.FaceDetectorOptions(base_options=mp_python.BaseOptions(model_asset_path=model_path))
)

def preprocess_for_model(img_rgb, margin_ratio=0.25):
    h, w = img_rgb.shape[:2]
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result = detector.detect(mp_image)
    if result.detections:
        bbox = result.detections[0].bounding_box
        mx, my = int(bbox.width * margin_ratio), int(bbox.height * margin_ratio)
        x1, y1 = max(0, bbox.origin_x - mx), max(0, bbox.origin_y - my)
        x2, y2 = min(w, bbox.origin_x + bbox.width + mx), min(h, bbox.origin_y + bbox.height + my)
        crop = img_rgb[y1:y2, x1:x2] if (x2 > x1 and y2 > y1) else img_rgb
        face_found = True
    else:
        crop, face_found = img_rgb, False
    resized = cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(resized, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    normalized = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)
    return normalized, face_found

LOCAL_BETA_PREPROCESSED = f"{LOCAL_ROOT}/scin_preprocessed"
os.makedirs(LOCAL_BETA_PREPROCESSED, exist_ok=True)

face_found_count = 0
for cid in usable_df["case_id"]:
    for ext in [".jpg", ".jpeg", ".png"]:
        src = os.path.join(LOCAL_BETA_IMAGES, cid + ext)
        if os.path.exists(src):
            img = np.array(Image.open(src).convert("RGB"))
            processed, found = preprocess_for_model(img)
            Image.fromarray(processed).save(os.path.join(LOCAL_BETA_PREPROCESSED, cid + ".jpg"), "JPEG", quality=92)
            face_found_count += int(found)
            break

print(f"Preprocessed {len(usable_df)} images. Face detected in {face_found_count}/{len(usable_df)}.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 4.7 MB/s eta 0:00:00
Preprocessed 4197 images. Face detected in 427/4197.


## Step 6 — Combine with existing data and fine-tune the Fitzpatrick head

Fine-tunes from Stage C's checkpoint (not from scratch), on the **union** of the original Fitzpatrick17k set
and the SCIN cases -- anchoring on what the model already learned while adding dermatologist-labeled examples
that skew toward darker skin types. A low learning rate and few epochs are deliberate: this is a correction
pass, not a full retrain.

In [ ]:
import tarfile, time
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
import tensorflow as tf

# Re-extract the original Fitzpatrick17k preprocessed set for the combined fine-tune
FITZ_PP_ARCHIVE = manifest["preprocessing"]["fitzpatrick17k"]["drive_archive"]
LOCAL_EXTRACTED = f"{LOCAL_ROOT}/extracted_preprocessed"
LOCAL_FITZ_DIR = f"{LOCAL_EXTRACTED}/fitzpatrick17k_preprocessed"
if not os.path.exists(LOCAL_FITZ_DIR):
    local_archive = "/content/aura_local/fitzpatrick17k_preprocessed.tar.gz"
    shutil.copy2(FITZ_PP_ARCHIVE, local_archive)
    with tarfile.open(local_archive, "r:gz") as tar:
        tar.extractall(LOCAL_EXTRACTED, filter="data")

original_labels = pd.read_csv(f"{LOCAL_FITZ_DIR}/processed_labels.csv")
original_labels["filepath"] = LOCAL_FITZ_DIR + "/" + original_labels["processed_filename"]
original_labels["fp_idx"] = original_labels["fitzpatrick_scale"].astype(int) - 1
original_labels["source"] = "fitzpatrick17k"
original_labels["is_asian"] = False

beta_labels = usable_df.copy()
beta_labels["filepath"] = LOCAL_BETA_PREPROCESSED + "/" + beta_labels["case_id"] + ".jpg"
beta_labels["fp_idx"] = beta_labels["fitzpatrick_scale"].astype(int) - 1
beta_labels["source"] = "scin"

combined_df = pd.concat([
    original_labels[["filepath", "fp_idx", "source", "is_asian"]],
    beta_labels[["filepath", "fp_idx", "source", "is_asian"]],
], ignore_index=True)
print(f"Combined fine-tuning set: {len(combined_df)} ({len(original_labels)} original + {len(beta_labels)} SCIN)")

combined_train, combined_val = train_test_split(
    combined_df, test_size=0.15, stratify=combined_df["fp_idx"], random_state=42
)
class_weights_arr = compute_class_weight("balanced", classes=np.arange(6), y=combined_train["fp_idx"].values)
class_weights = {i: w for i, w in enumerate(class_weights_arr)}
print("Recomputed class weights:", {k: round(v, 2) for k, v in class_weights.items()})


Combined fine-tuning set: 7931 (3734 original + 4197 SCIN)
Recomputed class weights: {0: np.float64(2.7), 1: np.float64(0.65), 2: np.float64(0.56), 3: np.float64(0.74), 4: np.float64(1.32), 5: np.float64(4.91)}


In [ ]:
def make_dataset(filepaths, labels, num_classes, training, batch_size=16):
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=num_classes)
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels_onehot))
    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = preprocess_input(tf.cast(img, tf.float32))
        return img, label
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(1024)
        ds = ds.map(lambda i, l: (tf.image.random_flip_left_right(i), l), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds_e = make_dataset(combined_train["filepath"].tolist(), combined_train["fp_idx"].tolist(), 6, training=True)
val_ds_e = make_dataset(combined_val["filepath"].tolist(), combined_val["fp_idx"].tolist(), 6, training=False)

# Single-output model: just backbone + fitzpatrick_output. Stage E never trains undertone anyway
# (it was only ever a zero-weighted placeholder) -- going single-output here sidesteps this Keras
# version's broken handling of class_weight on multi-output models entirely, rather than working
# around it. Same pattern Stage D already used successfully.
backbone = EfficientNetB0(include_top=False, weights=None, pooling="avg", input_shape=(IMG_SIZE, IMG_SIZE, 3))
fitzpatrick_head = layers.Dense(6, activation="softmax", name="fitzpatrick_output")(backbone.output)
stage_e_model = Model(inputs=backbone.input, outputs=fitzpatrick_head)

local_stage_c = "/content/aura_local/stage_c_best.weights.h5"
shutil.copy2(STAGE_C_CKPT, local_stage_c)
stage_e_model.load_weights(local_stage_c, skip_mismatch=True)
print("Loaded Stage C weights as the fine-tuning starting point (undertone head skipped -- not needed here).")

# Unfreeze everything for this correction pass, but at a much lower LR than Stage C used,
# since this is nudging an already-fine-tuned model, not training from a frozen/fresh state.
for layer in stage_e_model.layers:
    layer.trainable = True

stage_e_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

LOCAL_CKPT_DIR = f"{LOCAL_ROOT}/checkpoints"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
LOCAL_CKPT_E_BEST = f"{LOCAL_CKPT_DIR}/stage_e_best.weights.h5"
DRIVE_CKPT_E_BEST = f"{DRIVE_CKPT_DIR}/stage_e_best.weights.h5"

STAGE_E_EPOCHS = 8
best_val_acc_e = 0.0
for epoch in range(STAGE_E_EPOCHS):
    print(f"\n=== Stage E epoch {epoch+1}/{STAGE_E_EPOCHS} ===")
    history = stage_e_model.fit(train_ds_e, validation_data=val_ds_e, epochs=1, verbose=1,
                                 class_weight=class_weights)
    val_acc = history.history["val_accuracy"][0]
    if val_acc > best_val_acc_e:
        best_val_acc_e = val_acc
        stage_e_model.save_weights(LOCAL_CKPT_E_BEST)
        shutil.copy2(LOCAL_CKPT_E_BEST, DRIVE_CKPT_E_BEST)
        print(f"  ✅ New best ({val_acc:.3f}) synced to Drive")

print(f"\nStage E complete. Best combined-validation Fitzpatrick accuracy: {best_val_acc_e:.3f}")
print(f"(Compare to L1_T3's Stage C result of {manifest['training']['stage_c_finetuned_val_accuracy']:.3f} on")
print(f" Fitzpatrick17k alone -- this validation set includes held-out Filipino beta images too, so it's")
print(f" not a perfectly apples-to-apples comparison, but directionally informative.)")

Loaded Stage C weights as the fine-tuning starting point (undertone head skipped -- not needed here).

=== Stage E epoch 1/8 ===
422/422 ━━━━━━━━━━━━━━━━━━━━ 184s 219ms/step - accuracy: 0.3173 - loss: 1.4262 - val_accuracy: 0.3235 - val_loss: 1.5073
  ✅ New best (0.324) synced to Drive

=== Stage E epoch 2/8 ===
422/422 ━━━━━━━━━━━━━━━━━━━━ 32s 74ms/step - accuracy: 0.3273 - loss: 1.3837 - val_accuracy: 0.3202 - val_loss: 1.5054

=== Stage E epoch 3/8 ===
422/422 ━━━━━━━━━━━━━━━━━━━━ 33s 76ms/step - accuracy: 0.3372 - loss: 1.3463 - val_accuracy: 0.3252 - val_loss: 1.4958
  ✅ New best (0.325) synced to Drive

=== Stage E epoch 4/8 ===
422/422 ━━━━━━━━━━━━━━━━━━━━ 33s 76ms/step - accuracy: 0.3519 - loss: 1.3106 - val_accuracy: 0.3361 - val_loss: 1.4762
  ✅ New best (0.336) synced to Drive

=== Stage E epoch 5/8 ===
422/422 ━━━━━━━━━━━━━━━━━━━━ 33s 76ms/step - accuracy: 0.3600 - loss: 1.2835 - val_accuracy: 0.3403 - val_loss: 1.4725
  ✅ New best (0.340) synced to Drive

=== Stage E epoch

**Note on the `class_weight` call above:** this uses the same multi-output dict pattern flagged back in
`L1_T3` Step 6 as version-dependent. If it errors here, apply the same fix: drop `class_weight=` and either
compute per-sample weights manually or temporarily switch to a single-output model for this training pass.

## Step 7 — Evaluate on the Asian-flagged SCIN subset of the validation split

The combined validation set mixes sources. This cell isolates the Asian-flagged SCIN cases. Report the result
as "Asian subset of SCIN", **not** as Filipino accuracy: SCIN does not identify Filipino subjects.

In [ ]:
beta_val = combined_val[combined_val["is_asian"]]
print(f"Asian-flagged SCIN cases in the validation split: {len(beta_val)}")

if len(beta_val) == 0:
    print("⚠️  No Asian-flagged cases landed in validation by chance of the random split -- re-run Step 6 with a")
    print("   different random_state, or increase test_size, to get Asian-subset validation coverage.")
else:
    beta_val_ds = make_dataset(beta_val["filepath"].tolist(), beta_val["fp_idx"].tolist(), 6, training=False)
    stage_e_model.load_weights(LOCAL_CKPT_E_BEST)

    y_true_beta, y_pred_beta = [], []
    for imgs, labels in beta_val_ds:
        fp_preds = stage_e_model.predict(imgs, verbose=0)
        y_pred_beta.extend(np.argmax(fp_preds, axis=1).tolist())
        y_true_beta.extend(np.argmax(labels.numpy(), axis=1).tolist())

    from sklearn.metrics import accuracy_score
    exact = accuracy_score(y_true_beta, y_pred_beta)
    within_1 = np.mean(np.abs(np.array(y_true_beta) - np.array(y_pred_beta)) <= 1)
    print(f"Asian-subset (SCIN) validation -- exact accuracy: {exact:.3f} | within ±1 class: {within_1:.3f}")
    print("(Small-N result -- treat as directional. This is NOT Filipino-specific accuracy;")
    print(" a real Filipino benchmark still needs Filipino-specific labeled data.)")

Asian-flagged SCIN cases in the validation split: 13
Asian-subset (SCIN) validation -- exact accuracy: 0.308 | within ±1 class: 0.769
(Small-N result -- treat as directional. This is NOT Filipino-specific accuracy;
 a real Filipino benchmark still needs Filipino-specific labeled data.)


## Step 8 — Export the updated TFLite model, archive, and verify

In [ ]:
export_model = Model(inputs=stage_e_model.input,
                      outputs=stage_e_model.get_layer("fitzpatrick_output").output)
converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

LOCAL_TFLITE_V2 = f"{LOCAL_CKPT_DIR}/skin_tone_classifier_v2_scin.tflite"
with open(LOCAL_TFLITE_V2, "wb") as f:
    f.write(tflite_model)
print(f"Exported: {LOCAL_TFLITE_V2} ({os.path.getsize(LOCAL_TFLITE_V2)/1e6:.2f} MB)")

DRIVE_TFLITE_V2 = f"{DRIVE_CKPT_DIR}/skin_tone_classifier_v2_scin.tflite"
shutil.copy2(LOCAL_TFLITE_V2, DRIVE_TFLITE_V2)
assert os.path.getsize(LOCAL_TFLITE_V2) == os.path.getsize(DRIVE_TFLITE_V2)
print("✅ Synced to Drive. Original v1 (Fitzpatrick17k-only) export is left untouched for comparison/rollback.")


Saved artifact at '/tmp/tmptkrsfvuy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_720')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  132836378196688: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  132836378195344: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  132836557336272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132836557337040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132836557336848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132836557337808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132836557337232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132836557335504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132836557338000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132836557337424: TensorSpec(shape=(), dtype=tf.resource, name=

In [ ]:
drive.flush_and_unmount()
time.sleep(3)
drive.mount('/content/drive')

for label, path in [("Stage E checkpoint", DRIVE_CKPT_E_BEST), ("TFLite v2 (SCIN-tuned)", DRIVE_TFLITE_V2)]:
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"{status} {label}: {path}")


Mounted at /content/drive
✅ Stage E checkpoint: /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/checkpoints/stage_e_best.weights.h5
✅ TFLite v2 (SCIN-tuned): /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/checkpoints/skin_tone_classifier_v2_scin.tflite


In [ ]:
from datetime import datetime, timezone

manifest["scin_finetuning"] = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "data_source": DATA_SOURCE["name"],
    "data_license": DATA_SOURCE["license"],
    "scin_cases_used": len(beta_labels),
    "asian_flagged_cases_used": int(beta_labels["is_asian"].sum()),
    "combined_training_set_size": len(combined_df),
    "stage_e_val_accuracy_combined": round(float(best_val_acc_e), 4),
    "tflite_v2_export": DRIVE_TFLITE_V2,
    "note": "Fine-tuned from Stage C on Fitzpatrick17k + SCIN (dermatologist-estimated Fitzpatrick labels). "
            "Asian-subset metrics are NOT Filipino-specific. MST-E was not used for training.",
}
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)
print("Manifest updated.")

Manifest updated.


## Done — what to do next

- **Re-run `L1_T4`'s bias check against the new TFLite model** (`skin_tone_classifier_v2_scin.tflite`) and
  compare with the v1 numbers. MST-E remains the held-out test and was never trained on.
- Report Step 7's number as **"Asian subset of SCIN"**. A true Filipino benchmark would still need
  Filipino-specific labeled data collected later (with consent).
- Undertone is unchanged by this notebook (SCIN has no undertone labels).
- Before shipping, confirm the licenses of everything used for training (SCIN, Fitzpatrick17k, the Roboflow
  undertone set) cover Aura's intended commercial use.
- Then move to **`L1_T6` -- Final Validation & Export Package**.

## Task Summary — L1_T5: Fine-Tuning on Public, Dermatologist-Labeled Data (SCIN)

**Goal:** Originally scoped as fine-tuning on 200+ consented, dermatologist-labeled Filipino beta images per
the dev plan — but since that real-world data collection hasn't happened yet (still pending, genuine
logistics: consent process, dermatologist partnership), this notebook substitutes **SCIN (Skin Condition
Image Network)**, a public, dermatologist-labeled dataset from Google, as an interim way to add real clinical
labels and nudge the model, while keeping the original Filipino-specific goal explicitly open rather than
treating this as equivalent to it.

**Process:**
1. Recorded SCIN's license and attribution obligations up front (SCIN Data Use License; attribution,
   modification-disclosure, and no-re-identification requirements), and explicitly excluded MST-E from any
   training use — its terms permit evaluation only, consistent with how `L1_T4` already used it strictly as
   a bias-check, never as training data.
2. Loaded SCIN via Hugging Face (`google/scin`, 5,033 cases), kept only cases with a usable dermatologist
   Fitzpatrick label (4,370), and dropped 173 where dermatologists disagreed with each other — leaving 4,197
   usable labeled cases, 80 of them self-reported as Asian.
3. Preprocessed all 4,197 through the same face-detect-or-fallback + CLAHE pipeline as every other dataset in
   this series.
4. Combined with the original Fitzpatrick17k set and fine-tuned from `L1_T3`'s Stage C checkpoint.
5. Hit and resolved three real compatibility issues along the way: `load_weights(by_name=...)` isn't valid
   for Keras 3's native `.weights.h5` format (fixed by dropping `by_name`, keeping `skip_mismatch=True`);
   this Keras version's `class_weight` doesn't support the nested multi-output dict format at all (fixed by
   rebuilding Stage E as a **single-output** model — it never trained the undertone head anyway, so this cost
   nothing); and the downstream evaluation cell needed the same single-output simplification applied to it.

**Accomplished:**
- **4,197 SCIN cases** added to the training pool (combined set: 7,931 images), with real dermatologist
  Fitzpatrick labels across all 6 classes (I: 329, II: 1,373, III: 1,352, IV: 713, V: 362, VI: 68).
- **Face detected in only 427/4,197 (10.2%)** of SCIN images — even lower than Fitzpatrick17k's 17.1% — meaning
  SCIN is even more dominated by non-facial skin photos. The fallback-to-full-image strategy handled this
  correctly, same as before.
- Stage E (fine-tuned from Stage C) reached **0.355 combined-validation accuracy** over 8 epochs, improving
  steadily each epoch (0.324 → 0.355). This is **not a clear improvement over Stage C's 0.367** on
  Fitzpatrick17k alone, though the two numbers aren't strictly comparable since the validation set composition
  changed — worth treating as roughly flat, not a confirmed gain.
- **Asian-subset-specific validation (n=13, very small): 30.8% exact, 76.9% within ±1** — noticeably lower
  exact accuracy than the overall combined number, though the sample is too small to read much into that gap
  either way.
- Exported `skin_tone_classifier_v2_scin.tflite` (4.55 MB), archived and verified present on Drive after a
  full `flush_and_unmount` + remount. Original v1 export left untouched for rollback/comparison.
- Manifest updated with data source, license obligations, and results.

**What was not included, and why:**
- **Real Filipino-specific validation** — SCIN's 80 "Asian"-flagged cases (13 in this validation split) are a
  broad continental proxy, not Filipino-specific, and far too few to draw a reliable conclusion from. This
  notebook's Asian-subset number is directional at best, not a benchmark to cite anywhere official.
- **Any claim that this closes the gap `L1_T3`/`L1_T4` identified** — it doesn't. SCIN adds real clinical
  label diversity and more V/VI examples than Fitzpatrick17k had alone, but the combined-validation accuracy
  barely moved, and the original plan's actual target (200+ consented, dermatologist-labeled Filipino
  subjects) remains the real fix, not a stopgap that's now been satisfied.
- **MST-E as training data** — correctly kept out per its license terms; it remains evaluation-only, as used
  in `L1_T4`.
- **A production decision between v1 and v2 TFLite exports** — both now exist side by side; given v2 didn't
  show a clear win, that choice (or an ensemble/further-tuning approach) is still open.